## Cell 1 — Install Dependencies

In [22]:
import os, sys


os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

!pip install -q pymupdf pdfplumber pandas sentence-transformers transformers torch tavily-python

print("All dependencies installed.")

All dependencies installed.


## Cell 2 — Configuration (Edit These Variables)

In [24]:



import os


TAVILY_API_KEY = "YOUR_TAVILY_API_KEY_HERE"         

PDF_FOLDER = r"C:\Users\Hp\Downloads\BRSR Reports"


NLI_MODEL_NAME          = "cross-encoder/nli-roberta-base"
EMBED_MODEL_NAME        = "all-MiniLM-L6-v2"
CONTRADICTION_THRESHOLD = 0.82  
ENTAILMENT_THRESHOLD    = 0.75

print(f"PDF folder : {PDF_FOLDER}")
print(f"NLI model           : {NLI_MODEL_NAME}")
print("Configuration loaded.")


PDF folder : C:\Users\Hp\Downloads\BRSR Reports
NLI model           : cross-encoder/nli-roberta-base
Configuration loaded.


In [26]:

COMPANY_NAME_MAP = {
    "AXIS":       "Axis Bank",
    "AXIS BANK":  "Axis Bank",
    "NESTLE":     "Nestlé India",
    "NESTLE INDIA": "Nestlé India",
    "POWERGRID":  "Power Grid",
    "POWER GRID CORPORATION": "Power Grid",
    "NTPC LTD":   "NTPC",
    "HUL":        "HUL",
    "BHARTI AIRTEL": "Bharti Airtel",
}

def standardise_company_name(name):
    return COMPANY_NAME_MAP.get(name.strip().upper(), name.strip())

print("Company name map loaded.")

Company name map loaded.


## Cell 3 — Load Models

In [28]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import re
import time

from sentence_transformers import CrossEncoder, SentenceTransformer, util
from tavily import TavilyClient
import pymupdf  

print("Loading NLI cross-encoder...")
nli_model   = CrossEncoder(NLI_MODEL_NAME)

print("Loading sentence-embedding model...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

print("Connecting to Tavily...")
tavily = TavilyClient(api_key="YOUR_TAVILY_API_KEY_HERE")  


print("All models ready.")

Loading NLI cross-encoder...
Loading sentence-embedding model...
Connecting to Tavily...
All models ready.


## Cell 4 — Stage 1: Hybrid Claim Extraction

**Upgrade from original:**  
Old code returned a hardcoded fallback string when keywords weren't found, inflating the Contradiction rate.  
New code uses:
1. Keyword matching (Phase A)
2. Semantic similarity against an anchor phrase (Phase B) — catches indirect disclosures
3. Returns `INSUFFICIENT_DISCLOSURE` only when *both* phases fail

In [30]:


from sentence_transformers import util

import pdfplumber

def extract_penalty_table_structured(file_path):
    """
    Phase 0: Uses pdfplumber to extract the BRSR Principle 1/9 penalty table
    with column structure preserved. Returns a clean natural language summary
    of what the table actually says — distinguishing between nil rows and
    rows with actual amounts.

    Returns (claim_text, mode) or (None, None) if no table found.
    """
    penalty_keywords = ["penalty", "fine", "penalties", "fines"]
    nil_terms = {"nil", "na", "n/a", "", "-", "0", "null"}

    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                tables = page.extract_tables()
                for table in tables:
                    if not table:
                        continue

                    
                    page_text_lower = (page.extract_text() or "").lower()
                    in_principle_section = (
                        "principle 1" in page_text_lower
                        or "principle 9" in page_text_lower
                        or "essential indicator" in page_text_lower
                        or "monetary penalties" in page_text_lower
                    )
                    if not in_principle_section:
                        continue

                    
                    table_str = " ".join(
                        str(cell).lower()
                        for row in table for cell in row if cell
                    )
                    if not any(kw in table_str for kw in penalty_keywords):
                        continue

                   
                    actual_penalties = []
                    nil_categories   = []

                    for row in table:
                        if not row:
                            continue
                        cells = [str(c).strip() if c else "" for c in row]
                        row_text = " ".join(cells).lower()

                        
                        if any(h in row_text for h in [
                            "ngrbc principle", "name of the regulatory",
                            "brief of the case", "amount (in", "has an appeal",
                            "monetary", "non-monetary", "imprisonment", "punishment"
                        ]):
                            continue

                        
                        has_penalty_word = any(kw in row_text for kw in penalty_keywords)
                        has_amount = False
                        amount_str = ""

                        for cell in cells:
                            cell_clean = cell.replace(",", "").replace("₹", "").replace("`", "").strip()
                            
                            try:
                                val = float(cell_clean)
                                if val > 0:
                                    has_amount = True
                                    amount_str = cell
                                    break
                            except ValueError:
                                
                                if any(c in cell for c in ["₹", "Rs.", "Rs ", "INR", "Crore", "Lakh", "crore", "lakh"]):
                                    if not any(n in cell.lower() for n in nil_terms):
                                        has_amount = True
                                        amount_str = cell
                                        break

                        all_nil = all(
                            c.lower().strip() in nil_terms or c.strip() == ""
                            for c in cells[1:]  
                        )

                        if has_penalty_word and has_amount:
                            
                            non_nil_cells = [
                                c for c in cells
                                if c.strip() and c.lower().strip() not in nil_terms
                            ]
                            penalty_desc = " | ".join(non_nil_cells[:5])
                            actual_penalties.append(penalty_desc)

                        elif has_penalty_word and all_nil:
                            category = cells[0].strip() if cells[0] else "Penalty/Fine"
                            nil_categories.append(category)

                    
                    if actual_penalties:
                        penalty_list = "; ".join(actual_penalties)
                        claim = (
                            f"The company discloses the following monetary penalties "
                            f"imposed during the reporting period: {penalty_list}."
                        )
                        return claim, "table_structured_penalty"

                    elif nil_categories:
                        return (
                            "No monetary penalties or fines were imposed on the company "
                            "during the reporting period.",
                            "table_structured_nil"
                        )

    except Exception as e:
        pass  

    return None, None

STRICT_OUTCOME_TERMS = [
    "no penalty", "no penalties",
    "no fine", "no fines",
    "nil penalty", "nil penalties",
    "nil fine", "nil fines",
    "nil fines or penalties",
    "nil penalties or fines",
    "zero penalty", "zero penalties",
    "zero fine", "zero fines",
    "penalty imposed", "penalties imposed",
    "fine imposed", "fines imposed",
    "penalty levied", "penalties levied",
    "fine levied", "fines levied",
    "penalty of ₹", "penalty of rs", "penalty of inr",
    "fine of ₹", "fine of rs", "fine of inr",
    "monetary penalty", "monetary penalties",
    "monetary fine", "monetary fines",
    "compounding fee", "no monetary",
    "nil instances of penalty", "nil instances of fine"
]

INVALID_CONTEXT_TERMS = [
    "risk of", "may result", "could result",
    "potential penalty", "potential fine",
    "risk management", "indemnify", "indemnification",
    "award", "awarded", "review", "reviewed",
    "policy", "policies",
    "may be imposed", "could be imposed",
    "subject to penalty", "subject to fine",
    "in the event", "where applicable",
    "as applicable", "to the extent",
    "may include fines", "show cause",
    "settlement amount", "enforcement action",
    "adjudication order", "notice issued",
    "notice was issued", "proceedings",
    
    "provide details", "details of corrective",
    "any corrective action", "indicate whether",
    "whether any", "if yes", "if no",
    "s. no.", "s.no", "name of authority",
    "brief of the case", "has the company",
    "mention the number"
]

FINAL_INDICATORS = ["no", "nil", "zero", "imposed", "levied", "₹", "rs.", "inr"]

ANCHOR_NIL     = "no monetary penalties or fines were imposed on the company during the reporting period"
ANCHOR_PENALTY = "monetary penalties or fines were imposed on the company"

anchor_nil_emb     = embed_model.encode(ANCHOR_NIL,     convert_to_tensor=True)
anchor_penalty_emb = embed_model.encode(ANCHOR_PENALTY, convert_to_tensor=True)

SEMANTIC_THRESHOLD = 0.60  


def passes_all_gates(sentence):
    s = sentence.lower()
    gate_a = any(term in s for term in STRICT_OUTCOME_TERMS)
    gate_b = not any(term in s for term in INVALID_CONTEXT_TERMS)
    gate_c = any(ind in s for ind in FINAL_INDICATORS)
    return gate_a and gate_b and gate_c


def detect_and_normalize_table_row(sentence):
    """
    Detects structured BRSR table rows like:
        "Penalty/Fine  NIL  NA  0  NA"
        "Monetary Penalties  Nil  Nil  Nil"
    Returns a normalized natural language claim if detected, else None.

    This is critical: NLI models cannot interpret raw table cells.
    Converting to natural language dramatically improves NLI accuracy.
    """
    s = sentence.lower()
    has_penalty_word = any(w in s for w in ["penalty", "fine", "penalties", "fines"])
    has_nil_or_zero  = "nil" in s or " 0 " in s or "\t0" in s
    is_short         = len(sentence.split()) <= 25

    if has_penalty_word and has_nil_or_zero and is_short:
        # Normalize to clean natural language for NLI
        return "No monetary penalties or fines were imposed on the company during the reporting period."

    return None


def extract_claim_from_pdf(file_path):
    """
    Four-phase extractor (v5) — multi-sentence window upgrade:
    Instead of returning a single matched sentence, returns a
    3-sentence window (one before + match + one after) so the
    NLI model has full context around the penalty statement.

    Phase A — Table detection + normalization
    Phase B — Strict keyword + three gates (returns 3-sent window)
    Phase C — Semantic similarity fallback (returns 3-sent window)
    Phase D — INSUFFICIENT_DISCLOSURE
    """
    try:
        doc = pymupdf.open(file_path)
        full_text = ""
        for page in doc:
            full_text += page.get_text()
        doc.close()
    except Exception as e:
        return f"PDF_ERROR: {e}", "pdf_error"

    
    structured_claim, structured_mode = extract_penalty_table_structured(file_path)
    if structured_claim:
        return structured_claim, structured_mode

    sentences = re.split(r'(?<=[.!?])\s+', full_text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

    def get_window(sentences, idx):
        """Returns up to 3 sentences centred on idx, cleaned."""
        start = max(0, idx - 1)
        end   = min(len(sentences), idx + 2)
        window = " ".join(sentences[start:end])
        window = re.sub(r'[\n\r\t]+', ' ', window)
        window = re.sub(r' {2,}', ' ', window).strip()
        return window

    
    for sentence in sentences:
        normalized = detect_and_normalize_table_row(sentence)
        if normalized:
            return normalized, "table_normalized"

    
    for idx, sentence in enumerate(sentences):
        if passes_all_gates(sentence):
            window = get_window(sentences, idx)
            return window, "keyword_strict"

    
    candidates = [
        (i, s) for i, s in enumerate(sentences)
        if not any(term in s.lower() for term in INVALID_CONTEXT_TERMS)
        and any(w in s.lower() for w in ["penalty", "fine", "monetary",
                                          "sebi", "rbi", "ngt", "regulatory"])
        and len(s) > 20
    ]

    if candidates:
        texts      = [s for _, s in candidates]
        cand_embs  = embed_model.encode(texts, convert_to_tensor=True)
        scores_nil     = util.cos_sim(anchor_nil_emb,     cand_embs)[0]
        scores_penalty = util.cos_sim(anchor_penalty_emb, cand_embs)[0]
        combined       = (scores_nil + scores_penalty) / 2

        best_pos   = int(combined.argmax())
        best_score = float(combined[best_pos])

        if best_score >= SEMANTIC_THRESHOLD:
            original_idx = candidates[best_pos][0]
            best_sent    = sentences[original_idx]
            if any(ind in best_sent.lower() for ind in FINAL_INDICATORS):
                window = get_window(sentences, original_idx)
                return window, f"semantic_strict (score={best_score:.3f})"

    return "INSUFFICIENT_DISCLOSURE", "no_match"


print("Stage 1 extract_claim_from_pdf() upgraded to multi-sentence window (v5).")

print(f"  Outcome terms    : {len(STRICT_OUTCOME_TERMS)}")
print(f"  Reject terms     : {len(INVALID_CONTEXT_TERMS)}")
print(f"  Semantic threshold: {SEMANTIC_THRESHOLD}")
print(f"  Table normalization: ON")


Stage 1 extract_claim_from_pdf() upgraded to multi-sentence window (v5).
  Outcome terms    : 36
  Reject terms     : 43
  Semantic threshold: 0.6
  Table normalization: ON


## Cell 5 — Stage 2: Filtered Evidence Retrieval

**Upgrade from original:**  
Old code dumped all 5 Tavily results into one blob and ran NLI on the concatenation.  
New code:
1. Searches per-source (regulatory + news separately)
2. Filters each snippet for regulatory keywords AND year presence before passing to NLI
3. Keeps individual snippets as separate evidence units (not concatenated)

In [32]:


import json
import os
import time

CACHE_FILE = "evidence_cache.json"




def _load_cache():
    """Load the on-disk cache. Returns empty dict if file doesn't exist yet."""
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def _save_cache(cache):
    """Persist the cache to disk atomically."""
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


def _cache_key(company, year):
    return f"{company}||{year}"


def invalidate_cache_entry(company, year):
    """
    Force a fresh Tavily fetch for one firm-year on the next run.
    Call this manually before re-running if you know the cached result is bad.
    """
    cache = _load_cache()
    key   = _cache_key(company, year)
    if key in cache:
        del cache[key]
        _save_cache(cache)
        print(f"  [Cache] Invalidated: {key}")
    else:
        print(f"  [Cache] Key not found (nothing to invalidate): {key}")


def show_cache_status():
    """Print which firm-years are already cached vs still need a Tavily call."""
    cache = _load_cache()
    print(f"Cache file : {CACHE_FILE}")
    print(f"Cached entries : {len(cache)}")
    print()
    if cache:
        for key, val in sorted(cache.items()):
            n = len(val)
            print(f"  ✅  {key}  ({n} evidence results)")
    else:
        print("  (empty — no entries cached yet)")




EVIDENCE_FILTER_KEYWORDS = [
    "penalty", "fine", "penalised", "penalized", "non-compliance",
    "enforcement", "adjudication", "settlement", "compounding",
    "rbi", "sebi", "ngt", "court order", "ministry", "tribunal",
    "trai", "dot", "cpcb", "mca", "sat"
]

FY_YEAR_MAP = {
    2023: ["2022", "2023", "FY23", "FY 23", "2022-23", "FY2023"],
    2024: ["2023", "2024", "FY24", "FY 24", "2023-24", "FY2024"],
    2025: ["2024", "2025", "FY25", "FY 25", "2024-25", "FY2025"],
}

COMPANY_REGULATOR_MAP = {
    "HDFC Bank":     ["RBI", "SEBI"],
    "SBI":           ["RBI", "SEBI"],
    "ICICI Bank":    ["RBI", "SEBI"],
    "Axis Bank":     ["RBI", "SEBI"],
    "Reliance":      ["SEBI", "NGT", "CPCB"],
    "NTPC":          ["NGT", "CPCB", "SEBI", "CEA"],
    "Power Grid":    ["NGT", "CERC", "SEBI"],
    "Infosys":       ["SEBI", "SEC", "MCA"],
    "TCS":           ["SEBI", "MCA"],
    "Wipro":         ["SEBI", "MCA"],
    "Tata Motors":   ["SEBI", "NGT", "MCA"],
    "Maruti Suzuki": ["SEBI", "NGT", "MCA"],
    "HUL":           ["SEBI", "FSSAI", "MCA"],
    "Nestlé India":  ["SEBI", "FSSAI", "MCA"],
    "ITC":           ["SEBI", "FSSAI", "MCA"],
    "L&T":           ["SEBI", "NGT", "MCA"],
    "Bharti Airtel": ["TRAI", "DoT", "SEBI"],
}

DEFAULT_REGULATORS = ["SEBI", "RBI", "NGT", "MCA"]

FY_CALENDAR_RANGE = {
    2023: (2022, 2023),
    2024: (2023, 2024),
    2025: (2024, 2025),
}



def score_temporal_alignment(published_date_str, year):
    if not published_date_str:
        return 0.5
    try:
        from datetime import datetime
        pub_year = datetime.fromisoformat(published_date_str[:10]).year
    except Exception:
        return 0.5
    start_year, end_year = FY_CALENDAR_RANGE.get(year, (year - 1, year))
    if start_year <= pub_year <= end_year:
        return 1.0
    elif abs(pub_year - end_year) == 1:
        return 0.5
    else:
        return 0.0



COMPANY_NAME_VARIANTS = {
    "HDFC Bank":     ["hdfc bank", "hdfc"],
    "SBI":           ["sbi", "state bank of india", "state bank"],
    "ICICI Bank":    ["icici bank", "icici"],
    "Axis Bank":     ["axis bank", "axis"],
    "Reliance":      ["reliance industries", "reliance"],
    "NTPC":          ["ntpc"],
    "Power Grid":    ["power grid corporation", "powergrid", "pgcil",
                      "power grid of india"],
    "Infosys":       ["infosys"],
    "TCS":           ["tcs", "tata consultancy"],
    "Wipro":         ["wipro"],
    "Tata Motors":   ["tata motors"],
    "Maruti Suzuki": ["maruti suzuki", "maruti"],
    "HUL":           ["hul", "hindustan unilever"],
    "Nestlé India":  ["nestle india", "nestlé india", "nestle"],
    "ITC":           ["itc limited", "itc ltd", "itc's", "itc share",
                      "itc stock", "itc cigarette", "itc hotel", "itc agro"],
    "L&T":           ["l&t", "larsen & toubro", "larsen and toubro", "l and t"],
    "Bharti Airtel": ["bharti airtel", "airtel"],
}


def is_evidence_relevant(snippet_text, year, company=None):
    """
    Returns True if the snippet passes three gates:
    1. Contains a regulatory/penalty keyword
    2. Contains a year string matching the BRSR reporting period
    3. (NEW) Contains the company name — prevents unrelated penalty documents
       from being scored against a company's nil-disclosure claim.
       Gate 3 is skipped if company is None (backward-compatible).

    Fix: ITC 2023 and Power Grid 2023 were getting CONTRADICTION from
    completely unrelated documents (random PDFs containing penalty keywords
    and a matching year) because the old filter had no company name check.
    """
    text_lower = snippet_text.lower()

   
    has_regulatory = any(kw in text_lower for kw in EVIDENCE_FILTER_KEYWORDS)
    if not has_regulatory:
        return False

    
    year_strings = FY_YEAR_MAP.get(year, [str(year)])
    has_year = any(ys in snippet_text for ys in year_strings)
    if not has_year:
        return False

    
    if company is not None:
        variants = COMPANY_NAME_VARIANTS.get(company, [company.lower()])
        has_company = any(v in text_lower for v in variants)
        if not has_company:
            return False

    return True




def _fetch_from_tavily(company, year):
    """
    Raw Tavily call — identical to the original search_external_evidence().
    Only called when no cache entry exists.
    Returns the normalised filtered list (same format as before).
    """
    regulators = COMPANY_REGULATOR_MAP.get(company, DEFAULT_REGULATORS)
    reg_str    = " OR ".join(regulators)
    fy_str     = f"FY {year}"

    queries = [
        f"{company} penalty fine {reg_str} enforcement order {fy_str}",
        f"{company} regulatory penalty non-compliance fine {fy_str} "
        f"site:thehindu.com OR site:businessstandard.com OR "
        f"site:livemint.com OR site:indianexpress.com OR site:moneycontrol.com",
        f"{company} penalty {regulators[0]} {fy_str} site:rbi.org.in OR "
        f"site:sebi.gov.in OR site:ngt.gov.in OR site:mca.gov.in",
    ]

    raw_results = []
    for q in queries:
        try:
            resp = tavily.search(
                query=q,
                search_depth="advanced",
                max_results=5
            )
            raw_results.extend(resp.get("results", []))
            time.sleep(0.5)
        except Exception as e:
            print(f"  [Tavily Error] {e}")

    seen_urls      = set()
    unique_results = []
    for r in raw_results:
        url = r.get("url", "")
        if url not in seen_urls:
            seen_urls.add(url)
            unique_results.append(r)

    filtered = []
    for r in unique_results:
        content        = r.get("content", "")
        pub_date       = r.get("published_date", "")
        
        passed         = is_evidence_relevant(content, year, company=company)
        temporal_score = score_temporal_alignment(pub_date, year)
        if temporal_score == 0.0 and not passed:
            continue
        filtered.append({
            "content":        content,
            "url":            r.get("url", ""),
            "title":          r.get("title", ""),
            "published_date": pub_date,
            "temporal_score": temporal_score,
            "passed_filter":  passed
        })

    filtered.sort(
        key=lambda x: (x["passed_filter"], x["temporal_score"]),
        reverse=True
    )
    return filtered



def search_external_evidence(company, year):
    """
    Cache-aware evidence retrieval.

    1. Checks evidence_cache.json for an existing entry.
       If found  → returns cached results immediately (no Tavily call).
       If missing → calls Tavily, saves result to cache, then returns.

    The rest of the pipeline (Stage 3, Excel output) is completely unchanged —
    this function returns the exact same list-of-dicts format as before.
    """
    cache = _load_cache()
    key   = _cache_key(company, year)

    if key in cache:
        cached = cache[key]
        print(f"  [Cache HIT]  {key}  ({len(cached)} results, skipping Tavily)")
        return cached

    
    print(f"  [Cache MISS] {key}  → calling Tavily...")
    results = _fetch_from_tavily(company, year)

    
    cache[key] = results
    _save_cache(cache)
    print(f"  [Cache SAVE] {key}  ({len(results)} results written to {CACHE_FILE})")

    return results


def purge_empty_cache_entries():
    """
    Remove any cache entries that were saved with 0 results
    (e.g. from a run where Tavily was rate-limited throughout).
    These entries are poisoned — they will be re-fetched on the next run.
    """
    cache   = _load_cache()
    before  = len(cache)
    cleaned = {k: v for k, v in cache.items() if len(v) > 0}
    removed = before - len(cleaned)
    if removed > 0:
        _save_cache(cleaned)
        print(f"  [Cache] Purged {removed} empty entries. {len(cleaned)} valid entries remain.")
    else:
        print(f"  [Cache] No empty entries found. All {before} entries have results.")
    return cleaned



_startup_cache = _load_cache()
_empty = [k for k, v in _startup_cache.items() if len(v) == 0]
if _empty:
    _cleaned = {k: v for k, v in _startup_cache.items() if len(v) > 0}
    _save_cache(_cleaned)
    print(f"  [Cache] Auto-purged {len(_empty)} empty entries from previous failed run.")
    print(f"          These will be re-fetched from Tavily on next run.")
    _startup_cache = _cleaned

print("Stage 2 (v3 — cache-enabled) loaded.")
print(f"  Cache file       : {CACHE_FILE}")
print(f"  Valid cached entries : {len(_startup_cache)}")
print()
print("  Tip: call show_cache_status() to see which firms are cached.")
print("  Tip: call invalidate_cache_entry('Company', year) to force a re-fetch.")
print("  Tip: call purge_empty_cache_entries() to manually clear failed entries.")


Stage 2 (v3 — cache-enabled) loaded.
  Cache file       : evidence_cache.json
  Valid cached entries : 41

  Tip: call show_cache_status() to see which firms are cached.
  Tip: call invalidate_cache_entry('Company', year) to force a re-fetch.
  Tip: call purge_empty_cache_entries() to manually clear failed entries.


In [13]:

test_evidence = search_external_evidence("Axis Bank", 2023)

print(f"Total results: {len(test_evidence)}")
print(f"Passed filter: {sum(1 for e in test_evidence if e['passed_filter'])}")
print()
for e in test_evidence:
    print(f"  PASSED={e['passed_filter']} | {e['title']}")
    print(f"  URL: {e['url']}")
    print()

  [Cache HIT]  Axis Bank||2023  (14 results, skipping Tavily)
Total results: 14
Passed filter: 2

  PASSED=True | [PDF] Sandeep Poddar - London Stock Exchange
  URL: http://www.rns-pdf.londonstockexchange.com/rns/7264D_1-2024-9-11.pdf

  PASSED=True | Axis Bank, HDFC Bank Fined 2.91 Crore By RBI For Deficiencies In Regulatory Compliance
  URL: https://www.ndtv.com/business-news/axis-bank-hdfc-bank-fined-2-91-crore-by-rbi-for-deficiencies-in-regulatory-compliance-6538690

  PASSED=False | [PDF] A compilation of RBI penalties & enforcement actions in FY 24-25
  URL: https://faceofindia.org/wp-content/uploads/2025/04/A-compliation-of-RBI-penal-and-enforcement-actions-in-FY-24-25_-released-on-11-Apr-2025.pdf

  PASSED=False | rbi has imposed a monetary penalty on axis bank | ICICIdirect
  URL: https://www.icicidirect.com/research/equity/trending-news/rbi-has-imposed-a-monetary-penalty-on-axis-bank

  PASSED=False | RBI fines banks: RBI imposes penalties on five major banks for compliance f

## Cell 6 — Stage 3: Calibrated NLI Classification

**Upgrade from original:**  
Old code used raw argmax — whichever of the three NLI labels had highest probability won, even at 36% confidence.  
New code applies confidence thresholds:
- `contradiction_prob ≥ 0.75` → CONTRADICTION
- `entailment_prob ≥ 0.75` → ENTAILMENT  
- otherwise → NEUTRAL  

Contradiction-priority rule preserved: if *any* evidence unit crosses 0.75 for contradiction, the observation is CONTRADICTION.

In [34]:


NLI_LABEL_ORDER = ["contradiction", "entailment", "neutral"]


def classify_pair(claim, evidence_text):
    """NLI on one (claim, evidence) pair with calibrated thresholds."""
    scores = nli_model.predict([(claim, evidence_text)])
    logits = np.array(scores[0])
    probs  = np.exp(logits) / np.exp(logits).sum()

    p_contra  = float(probs[0])
    p_entail  = float(probs[1])
    p_neutral = float(probs[2])

    
    MIN_CONFIDENCE = 0.60
    if max(p_contra, p_entail, p_neutral) < MIN_CONFIDENCE:
        return "INSUFFICIENT_DISCLOSURE", 0.0, p_contra, p_entail, p_neutral

    if p_contra >= CONTRADICTION_THRESHOLD:
        label, conf = "CONTRADICTION", p_contra
    elif p_entail >= ENTAILMENT_THRESHOLD:
        label, conf = "ENTAILMENT", p_entail
    else:
        label, conf = "NEUTRAL", max(p_contra, p_entail, p_neutral)

    return label, conf, p_contra, p_entail, p_neutral


def claim_is_nil_type(claim):
    """Returns True if the claim asserts absence of penalty (nil/no/zero)."""
    c = claim.lower()
    return any(t in c for t in ["no penalty", "no fine", "no fines", "no penalties",
                                  "nil", "zero penalty", "zero fine",
                                  "not imposed", "were not", "no monetary"])


def claim_has_amount(claim):
    """Returns True if the claim contains a NON-ZERO monetary amount.

    Fix (Issue 1): previously returned True even when the claim said
    '₹0' or a nil-table row contained 'crore' in context, causing Rule 2
    to wrongly force CONTRADICTION on zero-value disclosures.
    Now explicitly excludes claims that also assert nil/zero/no penalty.
    Affected rows (pre-fix): Reliance 2025, Power Grid 2023/2024, Tata Motors 2024/2025.
    """
    c = claim.lower()
    
    has_currency = any(t in c for t in ["₹", "rs.", "rs ", "inr", "crore", "lakh"])
    if not has_currency:
        return False
    
    has_nil = any(t in c for t in ["nil", "zero", "no penalty", "no fine", "not imposed",
                                    "no monetary", "not levied", "no penalties", "no fines"])
    return not has_nil


def apply_rule_based_override(claim, label, confidence, detail_rows):
    """
    Two post-NLI rules to fix systematic neutral inflation:

    Rule 1 — Nil claim, no strong contradiction found → ENTAILMENT
        Logic: if the company says "no penalties" and external evidence
        does not strongly contradict this (no source crossed the 0.75
        contradiction threshold), absence is confirmed → ENTAILMENT.

    Rule 2 — Amount claim + NEUTRAL → CONTRADICTION
        Logic: a sentence like "Penalty of ₹1.30 crore imposed" is a
        concrete factual claim. NLI calling this NEUTRAL when evidence
        exists is a model failure. A specific disclosed amount almost
        always conflicts with a nil-penalty narrative → CONTRADICTION.
    """
    strongest_contradiction = max(
        (r["p_contra"] for r in detail_rows), default=0.0
    )

    # Rule 1
    if claim_is_nil_type(claim) and label == "NEUTRAL":
        if strongest_contradiction < CONTRADICTION_THRESHOLD:
            return "ENTAILMENT", confidence, "rule1_nil_confirmed"

    # Rule 2
    if claim_has_amount(claim) and label == "NEUTRAL":
        return "CONTRADICTION", confidence, "rule2_amount_present"

    return label, confidence, None




def get_claim_quality(claim, extraction_mode):
    """
    Classifies the quality of an extracted claim for the quality gate in Stage 4.
    Returns: CLEAR / PARTIAL / VAGUE / TABLE_FRAGMENT / NONE

    CLEAR         — clean nil or explicit penalty disclosure
    PARTIAL       — has some signal but incomplete
    VAGUE         — boilerplate / template / instruction text
    TABLE_FRAGMENT— truncated table row missing key fields
    NONE          — INSUFFICIENT_DISCLOSURE or empty
    """
    if not claim or claim == "INSUFFICIENT_DISCLOSURE" or claim.startswith("PDF_ERROR"):
        return "NONE"

    c = claim.lower().strip()

    
    if extraction_mode in ("table_structured_penalty", "table_structured_nil",
                           "table_normalized"):
        return "CLEAR"

    
    nil_phrases = ["no monetary penalties", "no penalties", "no fines",
                   "nil", "not imposed", "not levied", "no fine"]
    if any(p in c for p in nil_phrases):
        return "CLEAR"

    
    if any(t in c for t in ["₹", "rs.", "inr", "crore", "lakh"]):
        return "CLEAR"


    template_phrases = [
        "provide details", "if yes", "if no", "details of fines / penalties",
        "details of any corrective", "provide details of any",
        "name of the regulator", "has the entity", "whether",
        "in the following format", "yes/no"
    ]
    if any(p in c for p in template_phrases):
        return "VAGUE"

    
    if "|" in claim and len(claim) < 200:
        return "TABLE_FRAGMENT"
    if len(claim) < 80:
        return "TABLE_FRAGMENT"

    
    signal_phrases = ["penalty", "fine", "non-compliance", "enforcement",
                      "compounding", "settlement", "imposed", "levied"]
    if any(p in c for p in signal_phrases):
        return "PARTIAL"

    return "VAGUE"

def run_nli_on_evidence_list(claim, evidence_list):
    """
    Runs NLI on quality-filtered evidence, applies calibrated thresholds,
    then applies rule-based overrides for nil claims and amount claims.
    Returns (final_label, final_confidence, detail_rows).
    """
    if claim == "INSUFFICIENT_DISCLOSURE" or claim.startswith("PDF_ERROR"):
        return "INSUFFICIENT_DISCLOSURE", 0.0, []

    
    quality_evidence = [
        e for e in evidence_list
        if e["passed_filter"] and e.get("temporal_score", 0.5) > 0.0
    ]

    if not quality_evidence:
       
        if claim_is_nil_type(claim):
            return "ENTAILMENT", 0.0, []
        return "NO_EVIDENCE_FOUND", 0.0, []

    detail_rows    = []
    contradictions = []
    entailments    = []
    neutrals       = []

    for ev in quality_evidence:
        ev_text = ev["content"]
        if not ev_text.strip():
            continue

        label, conf, pc, pe, pn = classify_pair(claim, ev_text)

        row = {
            "label":        label,
            "confidence":   round(conf, 4),
            "p_contra":     round(pc, 4),
            "p_entail":     round(pe, 4),
            "p_neutral":    round(pn, 4),
            "source_url":   ev["url"],
            "source_title": ev["title"]
        }
        detail_rows.append(row)

        if label == "CONTRADICTION":        contradictions.append(row)
        elif label == "ENTAILMENT":          entailments.append(row)
        elif label == "INSUFFICIENT_DISCLOSURE": pass  
        else:                                neutrals.append(row)

    if not detail_rows:
        return "NO_EVIDENCE_FOUND", 0.0, []

   
    if not contradictions and not entailments and not neutrals:
        return "INSUFFICIENT_DISCLOSURE", 0.0, detail_rows

    if contradictions:
        strongest = max(contradictions, key=lambda x: x["confidence"])
        base_label, base_conf = "CONTRADICTION", strongest["confidence"]
    elif entailments:
        strongest = max(entailments, key=lambda x: x["confidence"])
        base_label, base_conf = "ENTAILMENT", strongest["confidence"]
    elif neutrals:
        strongest = max(neutrals, key=lambda x: x["confidence"])
        base_label, base_conf = "NEUTRAL", strongest["confidence"]
    else:
        return "INSUFFICIENT_DISCLOSURE", 0.0, detail_rows

   
    final_label, final_conf, override_used = apply_rule_based_override(
        claim, base_label, base_conf, detail_rows
    )

    if override_used:
        print(f"  [Override applied: {override_used}] {base_label} → {final_label}")

    return final_label, final_conf, detail_rows


print("Stage 3 (v4 + rule-based override) loaded.")
print(f"  Contradiction threshold : {CONTRADICTION_THRESHOLD}")
print(f"  Entailment threshold    : {ENTAILMENT_THRESHOLD}")
print(f"  Rule 1 (nil → entailment)  : ON")
print(f"  Rule 2 (amount → contradiction) : ON")

Stage 3 (v4 + rule-based override) loaded.
  Contradiction threshold : 0.82
  Entailment threshold    : 0.75
  Rule 1 (nil → entailment)  : ON
  Rule 2 (amount → contradiction) : ON


## Cell 7 — Stage 4 & 5: Main Audit Function + Structured Output

In [36]:


def find_pdf(company_name, year, pdf_folder):
    """Fuzzy match: find the PDF for a given company + year in the folder."""
    all_pdfs = [f for f in os.listdir(pdf_folder) if f.lower().endswith(".pdf")]
    term = company_name.lower().replace(" ", "").replace("&", "and")
    matches = [
        f for f in all_pdfs
        if term in f.lower().replace(" ", "").replace("-", "").replace("_", "").replace("&", "and")
        and str(year) in f
    ]
    return os.path.join(pdf_folder, matches[0]) if matches else None


def run_audit(target_list, pdf_folder):
    """
    target_list: list of (company_name, year) tuples
    pdf_folder:  path to folder containing BRSR PDFs

    Returns a DataFrame with one row per firm-year observation.
    """
    import os

    records      = []   
    detail_log   = []   

    for company, year in target_list:
        print(f"\n{'='*60}")
        print(f"  AUDITING: {company} ({year})")
        print(f"{'='*60}")

        # ---- Find PDF ----
        pdf_path = find_pdf(company, year, pdf_folder)
        if not pdf_path:
            print(f"  [SKIP] PDF not found for {company} {year}")
            records.append({
                "Company":         company,
                "Year":            year,
                "Claim":           "PDF_NOT_FOUND",
                "Extraction_Mode": "n/a",
                "Evidence_Count_Raw":      0,
                "Evidence_Count_Filtered": 0,
                "Final_Label":     "PDF_NOT_FOUND",
                "Confidence":      0.0,
                "Top_Source_URL":  "",
                "Top_Source_Title": ""
            })
            continue

        
        claim, extraction_mode = extract_claim_from_pdf(pdf_path)
        print(f"  Claim ({extraction_mode}):")
        print(f"    {claim[:120]}..." if len(claim) > 120 else f"    {claim}")

        
        evidence_list = search_external_evidence(company, year)
        n_raw      = len(evidence_list)
        n_filtered = sum(1 for e in evidence_list if e["passed_filter"])
        print(f"  Evidence: {n_raw} raw results, {n_filtered} passed quality filter")

        
        WEAK_QUALITY_LABELS = {"TABLE_FRAGMENT", "VAGUE", "NONE"}
        claim_quality_label = get_claim_quality(claim, extraction_mode)
        if claim_quality_label in WEAK_QUALITY_LABELS:
            print(f"  [Quality Gate] Claim quality '{claim_quality_label}' → "
                  f"skipping NLI, forcing INSUFFICIENT_DISCLOSURE")
            final_label  = "INSUFFICIENT_DISCLOSURE"
            confidence   = 0.0
            nli_details  = []
        else:
            final_label, confidence, nli_details = run_nli_on_evidence_list(claim, evidence_list)
        print(f"  Final Label: {final_label}  |  Confidence: {confidence:.4f}")

       
        top_src = nli_details[0] if nli_details else {}

        records.append({
            "Company":         company,
            "Year":            year,
            "Claim":           claim,
            "Extraction_Mode": extraction_mode,
            "Evidence_Count_Raw":      n_raw,
            "Evidence_Count_Filtered": n_filtered,
            "Final_Label":     final_label,
            "Confidence":      round(confidence, 4),
            "Top_Source_URL":  top_src.get("source_url", ""),
            "Top_Source_Title": top_src.get("source_title", "")
        })

       
        for d in nli_details:
            detail_log.append({
                "Company":   company,
                "Year":      year,
                "Claim":     claim[:200],
                "Evidence":  d.get("source_title", ""),
                "Source_URL": d.get("source_url", ""),
                "Pair_Label":  d["label"],
                "Confidence":  d["confidence"],
                "P_Contradiction": d["p_contra"],
                "P_Entailment":    d["p_entail"],
                "P_Neutral":       d["p_neutral"]
            })

        time.sleep(1.5)   

    results_df = pd.DataFrame(records)
    detail_df  = pd.DataFrame(detail_log)
    return results_df, detail_df


print("Stage 4 (Main Audit Runner) defined.")

Stage 4 (Main Audit Runner) defined.


## Cell 8 — Run the Full Audit

In [38]:


import os

# ── Company name standardisation ──
COMPANY_NAME_MAP = {
    "AXIS":                      "Axis Bank",
    "AXIS BANK":                 "Axis Bank",
    "NESTLE":                    "Nestlé India",
    "NESTLE INDIA":              "Nestlé India",
    "NESTLÉ INDIA":              "Nestlé India",
    "NESTLÉ":                    "Nestlé India",
    "POWERGRID":                 "Power Grid",
    "POWER GRID":                "Power Grid",
    "POWER GRID CORPORATION":    "Power Grid",
    "NTPC LTD":                  "NTPC",
    "HUL":                       "HUL",
    "BHARTI AIRTEL":             "Bharti Airtel",
    "TATA MOTORS":               "Tata Motors",
    "MARUTI SUZUKI":             "Maruti Suzuki",
    "MARUTI":                    "Maruti Suzuki",
    "HDFC BANK":                 "HDFC Bank",
    "ICICI BANK":                "ICICI Bank",
    "INFOSYS":                   "Infosys",
    "RELIANCE":                  "Reliance",
    "WIPRO":                     "Wipro",
    "TCS":                       "TCS",
    "ITC":                       "ITC",
    "SBI":                       "SBI",
    "L&T":                       "L&T",
}

def standardise_company_name(name):
    return COMPANY_NAME_MAP.get(name.strip().upper(), name.strip())

# ── Scope filter: only FY 2023, 2024, 2025 ──
VALID_YEARS = {2023, 2024, 2025}

# ── Auto-detect all PDFs from folder ──
raw_targets = []

for file in os.listdir(PDF_FOLDER):
    if file.lower().endswith(".pdf"):
        filename = file.replace(".pdf", "").replace(".PDF", "").strip()

        if "-" in filename:
            company_raw, year_raw = filename.rsplit("-", 1)

            if year_raw.strip().isdigit():
                year = int(year_raw.strip())

                
                if year not in VALID_YEARS:
                    continue

                
                company = standardise_company_name(company_raw.strip())
                raw_targets.append((company, year, os.path.join(PDF_FOLDER, file)))


targets = sorted(raw_targets, key=lambda x: (x[0], x[1]))

print(f"\nTotal firm-year observations (after year + name filter): {len(targets)}")
print(f"Years in scope: {sorted(VALID_YEARS)}")
print(f"\nAll detected entries:")
for t in targets:
    print(f"  {t[0]} ({t[1]})")



def _audit_one(company, year, pdf_path, records, detail_log):
    if not pdf_path:
        print(f"  [SKIP] PDF not found for {company} {year}")
        records.append({
            "Company": company, "Year": year,
            "Claim": "PDF_NOT_FOUND", "Extraction_Mode": "n/a",
            "Evidence_Count_Raw": 0, "Evidence_Count_Filtered": 0,
            "Final_Label": "PDF_NOT_FOUND", "Confidence": 0.0,
            "Top_Source_URL": "", "Top_Source_Title": ""
        })
        return records, detail_log

    
    claim, extraction_mode = extract_claim_from_pdf(pdf_path)
    print(f"  Claim ({extraction_mode}):")
    print(f"    {claim[:120]}..." if len(claim) > 120 else f"    {claim}")

    evidence_list = search_external_evidence(company, year)
    n_raw      = len(evidence_list)
    n_filtered = sum(1 for e in evidence_list if e["passed_filter"])
    print(f"  Evidence: {n_raw} raw, {n_filtered} passed filter")

   
    final_label, confidence, nli_details = run_nli_on_evidence_list(claim, evidence_list)
    print(f"  Final Label: {final_label}  |  Confidence: {confidence:.4f}")

    top_src = nli_details[0] if nli_details else {}

    records.append({
        "Company":                 company,
        "Year":                    year,
        "Claim":                   claim,
        "Extraction_Mode":         extraction_mode,
        "Evidence_Count_Raw":      n_raw,
        "Evidence_Count_Filtered": n_filtered,
        "Final_Label":             final_label,
        "Confidence":              round(confidence, 4),
        "Top_Source_URL":          top_src.get("source_url", ""),
        "Top_Source_Title":        top_src.get("source_title", "")
    })

    for d in nli_details:
        detail_log.append({
            "Company":         company,
            "Year":            year,
            "Claim":           claim[:200],
            "Evidence":        d.get("source_title", ""),
            "Source_URL":      d.get("source_url", ""),
            "Pair_Label":      d["label"],
            "Confidence":      d["confidence"],
            "P_Contradiction": d["p_contra"],
            "P_Entailment":    d["p_entail"],
            "P_Neutral":       d["p_neutral"]
        })

    import time
    time.sleep(1.5)

    return records, detail_log



def run_full_audit():
    records    = []
    detail_log = []

    for company, year, pdf_path in targets:
        print(f"\n{'='*60}\n  AUDITING: {company} ({year})\n{'='*60}")
        records, detail_log = _audit_one(company, year, pdf_path, records, detail_log)

    results_df = pd.DataFrame(records)

    
    DETAIL_COLUMNS = [
        "Company", "Year", "Claim", "Evidence", "Source_URL",
        "Pair_Label", "Confidence", "P_Contradiction", "P_Entailment", "P_Neutral"
    ]
    if detail_log:
        detail_df = pd.DataFrame(detail_log)
        
        for col in DETAIL_COLUMNS:
            if col not in detail_df.columns:
                detail_df[col] = ""
    else:
        detail_df = pd.DataFrame(columns=DETAIL_COLUMNS)

    return results_df, detail_df


# ── Execute ──
results_df, detail_df = run_full_audit()




import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

import re

ILLEGAL_CHARS_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f]')

def clean_claim_for_display(claim_text):
    if claim_text in ("INSUFFICIENT_DISCLOSURE", "PDF_NOT_FOUND") or \
       claim_text.startswith("PDF_ERROR"):
        return claim_text
    
    cleaned = ILLEGAL_CHARS_RE.sub('', claim_text)
    
    cleaned = re.sub(r'[\n\r\t]+', ' ', cleaned)
    cleaned = re.sub(r' {2,}', ' ', cleaned).strip()
    
    return cleaned[:32767]


def get_claim_quality(claim, extraction_mode):
    if claim in ("INSUFFICIENT_DISCLOSURE", "PDF_NOT_FOUND") or \
       claim.startswith("PDF_ERROR"):
        return "NONE"
    if extraction_mode == "table_normalized":
        return "CLEAR"
    if extraction_mode == "table_detected":
        return "TABLE_FRAGMENT"
    c = claim.lower()
    has_specific = any(w in c for w in ["rbi", "sebi", "ngt", "₹", "rs.", "crore",
                                         "lakh", "2023", "2024", "2025", "fy"])
    has_outcome  = any(w in c for w in ["no penalty", "no fine", "nil", "zero",
                                         "imposed", "levied", "not imposed"])
    if has_specific and has_outcome:
        return "CLEAR"
    elif has_outcome:
        return "PARTIAL"
    else:
        return "VAGUE"


def get_confidence_bucket(confidence, label):
    if label in ("INSUFFICIENT_DISCLOSURE", "PDF_NOT_FOUND", "NO_EVIDENCE_FOUND"):
        return "N/A"
    if confidence >= 0.80:
        return "High (>0.80)"
    elif confidence >= 0.60:
        return "Medium (0.60–0.80)"
    else:
        return "Low (<0.60)"


def get_override_description(claim, final_label):
    if claim in ("INSUFFICIENT_DISCLOSURE", "PDF_NOT_FOUND"):
        return "No penalty sentence found in BRSR PDF"
    if claim_is_nil_type(claim) and final_label == "ENTAILMENT":
        return "Rule 1: Nil/No-penalty claim + no strong contradiction found → ENTAILMENT"
    if claim_has_amount(claim) and final_label == "CONTRADICTION":
        return "Rule 2: Specific monetary amount in claim → CONTRADICTION"
    if final_label == "CONTRADICTION":
        return "NLI model: claim semantically conflicts with external evidence (conf ≥ 0.75)"
    if final_label == "ENTAILMENT":
        return "NLI model: claim semantically consistent with external evidence (conf ≥ 0.75)"
    if final_label == "NEUTRAL":
        return "NLI model: evidence found but no confident contradiction or entailment"
    if final_label == "NO_EVIDENCE_FOUND":
        return "No evidence passed the year + regulatory keyword quality filter"
    return ""


def get_evidence_summary(company, year, detail_df):
    subset = detail_df[
        (detail_df["Company"] == company) &
        (detail_df["Year"] == year)
    ]
    if subset.empty:
        return "", ""
    titles = subset["Evidence"].dropna().tolist()
    urls   = subset["Source_URL"].dropna().tolist()
    t1 = titles[0] if len(titles) > 0 else ""
    t2 = titles[1] if len(titles) > 1 else ""
    u1 = urls[0]   if len(urls)   > 0 else ""
    u2 = urls[1]   if len(urls)   > 1 else ""
    return f"{t1} | {u1}", f"{t2} | {u2}"


def get_time_alignment(year, detail_df, company):
    subset = detail_df[
        (detail_df["Company"] == company) &
        (detail_df["Year"] == year)
    ]
    if subset.empty:
        return "UNCLEAR"
    year_strings = FY_YEAR_MAP.get(year, [str(year)])
    aligned = 0
    for ev_title in subset["Evidence"].dropna():
        if any(ys in str(ev_title) for ys in year_strings):
            aligned += 1
    if aligned == len(subset):
        return "YES"
    elif aligned > 0:
        return "PARTIAL"
    else:
        return "UNCLEAR"


def build_main_sheet(ws, results_df, detail_df):
    COLOURS = {
        "CONTRADICTION":           "FF6B6B",
        "ENTAILMENT":              "6BCB77",
        "INSUFFICIENT_DISCLOSURE": "FFD93D",
        "NEUTRAL":                 "AED6F1",
        "NO_EVIDENCE_FOUND":       "D5DBDB",
        "PDF_NOT_FOUND":           "F0F0F0",
    }
    CLAIM_QUALITY_COLOURS = {
        "CLEAR":          "D5F5E3",
        "PARTIAL":        "FEF9E7",
        "VAGUE":          "FDEDEC",
        "TABLE_FRAGMENT": "EBF5FB",
        "NONE":           "F2F3F4",
    }
    headers = [
        "Company", "Year",
        "Final_Label", "Confidence", "Confidence_Bucket",
        "Extraction_Mode", "Claim_Text", "Claim_Quality",
        "Decision_Reason",
        "Evidence_1_Title_and_URL", "Evidence_2_Title_and_URL",
        "Time_Aligned",
        "Evidence_Raw", "Evidence_Filtered",
        "Manual_Label", "Model_Correct (Yes/No)", "Error_Type", "Validator_Notes",
    ]
    header_fill  = PatternFill("solid", fgColor="1C2833")
    header_font  = Font(bold=True, color="FFFFFF", size=10)
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    thin         = Side(style="thin", color="CCCCCC")
    border       = Border(left=thin, right=thin, top=thin, bottom=thin)

    for col_idx, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = header_align
    ws.row_dimensions[1].height = 35

    legend_texts = {
        15: "CONTRADICTION / ENTAILMENT / NEUTRAL / INSUFFICIENT_DISCLOSURE",
        16: "Yes / No",
        17: "Extraction Error / Evidence Issue / NLI Misclassification / Rule-Based Bias / Correct",
        18: "Your notes on why you agreed or disagreed with the model",
    }
    legend_fill = PatternFill("solid", fgColor="ECF0F1")
    legend_font = Font(italic=True, size=8, color="7F8C8D")
    for col_idx in range(1, len(headers) + 1):
        cell = ws.cell(row=2, column=col_idx, value=legend_texts.get(col_idx, ""))
        cell.fill      = legend_fill
        cell.font      = legend_font
        cell.alignment = Alignment(vertical="center", wrap_text=True)
    ws.row_dimensions[2].height = 25

    for row_idx, row in enumerate(results_df.itertuples(), 3):
        company    = row.Company
        year       = row.Year
        label      = row.Final_Label
        confidence = row.Confidence
        claim_raw  = row.Claim
        ext_mode   = row.Extraction_Mode
        ev_raw     = row.Evidence_Count_Raw
        ev_filt    = row.Evidence_Count_Filtered

        claim_clean     = clean_claim_for_display(claim_raw)
        claim_quality   = get_claim_quality(claim_raw, ext_mode)
        conf_bucket     = get_confidence_bucket(confidence, label)
        decision_reason = get_override_description(claim_raw, label)
        ev1, ev2        = get_evidence_summary(company, year, detail_df)
        time_aligned    = get_time_alignment(year, detail_df, company)

        row_data = [
            company, year,
            label, confidence, conf_bucket,
            ext_mode, claim_clean, claim_quality,
            decision_reason,
            ev1, ev2,
            time_aligned,
            ev_raw, ev_filt,
            "", "", "", "",
        ]

        for col_idx, value in enumerate(row_data, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.border    = border
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            if col_idx == 3:
                cell.fill = PatternFill("solid", fgColor=COLOURS.get(label, "FFFFFF"))
                cell.font = Font(bold=True, size=10)
            if col_idx == 8:
                cell.fill = PatternFill("solid",
                    fgColor=CLAIM_QUALITY_COLOURS.get(claim_quality, "FFFFFF"))
                cell.font = Font(bold=True, size=9)
            if col_idx == 12:
                ta_colour = {"YES": "D5F5E3", "PARTIAL": "FEF9E7",
                             "UNCLEAR": "FDEDEC"}.get(time_aligned, "FFFFFF")
                cell.fill = PatternFill("solid", fgColor=ta_colour)
            if col_idx in (15, 16, 17, 18):
                cell.fill = PatternFill("solid", fgColor="FFFDE7")

    col_widths = [16, 6, 24, 11, 18, 16, 55, 16, 45, 55, 55, 14, 10, 10, 24, 18, 28, 35]
    for i, width in enumerate(col_widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = width
    ws.freeze_panes = "C3"


def build_detail_sheet(ws, detail_df):
    headers = [
        "Company", "Year", "Claim_Text",
        "Evidence_Title", "Evidence_URL", "Pair_Label",
        "Confidence", "P_Contradiction", "P_Entailment", "P_Neutral"
    ]
    header_fill = PatternFill("solid", fgColor="1A5276")
    header_font = Font(bold=True, color="FFFFFF", size=11)
    for col_idx, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30

    PAIR_COLOURS = {"CONTRADICTION": "FF6B6B", "ENTAILMENT": "6BCB77", "NEUTRAL": "AED6F1"}
    thin   = Side(style="thin", color="CCCCCC")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for row_idx, row in enumerate(detail_df.itertuples(), 2):
        claim_clean = clean_claim_for_display(str(row.Claim))
        row_data = [
            row.Company, row.Year, claim_clean,
            row.Evidence, row.Source_URL, row.Pair_Label,
            row.Confidence, row.P_Contradiction, row.P_Entailment, row.P_Neutral,
        ]
        for col_idx, value in enumerate(row_data, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.border    = border
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            if col_idx == 6:
                cell.fill = PatternFill("solid",
                    fgColor=PAIR_COLOURS.get(row.Pair_Label, "FFFFFF"))
                cell.font = Font(bold=True)

    col_widths = [18, 6, 65, 45, 55, 16, 12, 16, 14, 12]
    for i, width in enumerate(col_widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = width
    ws.freeze_panes = "A2"


def build_summary_sheet(ws, results_df):
    ws.sheet_view.showGridLines = False
    title_font  = Font(bold=True, size=14, color="2C3E50")
    header_font = Font(bold=True, size=11, color="FFFFFF")
    header_fill = PatternFill("solid", fgColor="2C3E50")
    thin        = Side(style="thin", color="CCCCCC")
    border      = Border(left=thin, right=thin, top=thin, bottom=thin)

    ws["A1"] = "BRSR Forensic Audit — Summary Statistics"
    ws["A1"].font = title_font
    ws["A2"] = "Thesis: Beyond the Tick-Box | Ayush Kulhari | M2024ANLT007 | TISS Mumbai"
    ws["A2"].font = Font(italic=True, size=10, color="7F8C8D")
    ws.merge_cells("A1:E1")
    ws.merge_cells("A2:E2")

    ws["A4"] = "Overall Classification Distribution"
    ws["A4"].font = Font(bold=True, size=12)
    for col, hdr in [(1, "Classification"), (2, "Count"), (3, "Percentage")]:
        cell = ws.cell(row=5, column=col, value=hdr)
        cell.fill = header_fill
        cell.font = header_font

    COLOURS = {
        "CONTRADICTION":           "FF6B6B",
        "ENTAILMENT":              "6BCB77",
        "INSUFFICIENT_DISCLOSURE": "FFD93D",
        "NEUTRAL":                 "AED6F1",
        "NO_EVIDENCE_FOUND":       "D5DBDB",
    }
    counts = results_df["Final_Label"].value_counts()
    total  = len(results_df)

    for i, (label, count) in enumerate(counts.items(), 6):
        pct = count / total * 100
        ws.cell(row=i, column=1, value=label).fill  = PatternFill("solid", fgColor=COLOURS.get(label, "FFFFFF"))
        ws.cell(row=i, column=1).font   = Font(bold=True)
        ws.cell(row=i, column=1).border = border
        ws.cell(row=i, column=2, value=count).border  = border
        ws.cell(row=i, column=2).alignment = Alignment(horizontal="center")
        ws.cell(row=i, column=3, value=f"{pct:.1f}%").border = border
        ws.cell(row=i, column=3).alignment = Alignment(horizontal="center")

    risk_row   = 6 + len(counts) + 1
    risk_count = results_df["Final_Label"].isin(["INSUFFICIENT_DISCLOSURE", "CONTRADICTION"]).sum()
    risk_pct   = risk_count / total * 100
    ws.cell(row=risk_row, column=1, value="TRANSPARENCY RISK RATE").font = Font(bold=True, size=12)
    ws.cell(row=risk_row, column=2, value=f"{risk_count}/{total}").font  = Font(bold=True)
    ws.cell(row=risk_row, column=3, value=f"{risk_pct:.1f}%").font       = Font(bold=True, color="C0392B")
    for c in [1, 2, 3]:
        ws.cell(row=risk_row, column=c).border = border

    ct_row = risk_row + 3
    ws.cell(row=ct_row - 1, column=1, value="Company-Level Distribution").font = Font(bold=True, size=12)
    crosstab = pd.crosstab(results_df["Company"], results_df["Final_Label"])
    ws.cell(row=ct_row, column=1, value="Company").fill = header_fill
    ws.cell(row=ct_row, column=1).font   = header_font
    ws.cell(row=ct_row, column=1).border = border
    for j, col_name in enumerate(crosstab.columns, 2):
        cell = ws.cell(row=ct_row, column=j, value=col_name)
        cell.fill   = PatternFill("solid", fgColor=COLOURS.get(col_name, "CCCCCC"))
        cell.font   = Font(bold=True)
        cell.border = border
        cell.alignment = Alignment(horizontal="center")
    for i, (company, row_data) in enumerate(crosstab.iterrows(), ct_row + 1):
        ws.cell(row=i, column=1, value=company).border = border
        ws.cell(row=i, column=1).font = Font(bold=True)
        for j, val in enumerate(row_data, 2):
            c = ws.cell(row=i, column=j, value=int(val))
            c.border    = border
            c.alignment = Alignment(horizontal="center")

    ws.column_dimensions["A"].width = 30
    ws.column_dimensions["B"].width = 12
    ws.column_dimensions["C"].width = 14
    for col in ["D", "E", "F"]:
        ws.column_dimensions[col].width = 14



OUTPUT_PATH = "BRSR_Audit_Final_Report.xlsx"
wb = openpyxl.Workbook()

ws_main    = wb.active
ws_main.title = "Audit Results"
build_main_sheet(ws_main, results_df, detail_df)

ws_detail  = wb.create_sheet("Evidence Detail")
build_detail_sheet(ws_detail, detail_df)

ws_summary = wb.create_sheet("Summary")
build_summary_sheet(ws_summary, results_df)

wb.move_sheet("Summary", offset=-2)
wb.save(OUTPUT_PATH)

print(f"\n✅ Excel report saved: {OUTPUT_PATH}")
print(f"   Total observations : {len(results_df)}")
print(f"   Years covered      : {sorted(results_df['Year'].unique())}")
print(f"   Companies          : {sorted(results_df['Company'].unique())}")
print(f"\n   Label distribution:")
print(results_df['Final_Label'].value_counts().to_string())



Total firm-year observations (after year + name filter): 41
Years in scope: [2023, 2024, 2025]

All detected entries:
  Axis Bank (2023)
  Axis Bank (2024)
  Bharti Airtel (2023)
  Bharti Airtel (2024)
  Bharti Airtel (2025)
  HDFC Bank (2023)
  HDFC Bank (2024)
  HDFC Bank (2025)
  HUL (2023)
  HUL (2024)
  HUL (2025)
  ICICI Bank (2023)
  ICICI Bank (2024)
  ITC (2023)
  ITC (2024)
  Infosys (2023)
  Infosys (2024)
  Infosys (2025)
  L&T (2023)
  L&T (2024)
  L&T (2025)
  Maruti Suzuki (2023)
  Maruti Suzuki (2024)
  NTPC (2023)
  NTPC (2024)
  Nestlé India (2023)
  Nestlé India (2024)
  Power Grid (2023)
  Power Grid (2024)
  Reliance (2023)
  Reliance (2024)
  Reliance (2025)
  SBI (2024)
  SBI (2025)
  TCS (2023)
  TCS (2024)
  Tata Motors (2023)
  Tata Motors (2024)
  Tata Motors (2025)
  Wipro (2023)
  Wipro (2024)

  AUDITING: Axis Bank (2023)
  Claim (no_match):
    INSUFFICIENT_DISCLOSURE
  [Cache HIT]  Axis Bank||2023  (14 results, skipping Tavily)
  Evidence: 14 raw, 2 pas

Cannot set gray stroke color because /'P0' is an invalid float value
Cannot set gray stroke color because /'P0' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value


  Claim (table_normalized):
    No monetary penalties or fines were imposed on the company during the reporting period.
  [Cache HIT]  HDFC Bank||2023  (14 results, skipping Tavily)
  Evidence: 14 raw, 3 passed filter
  [Override applied: rule1_nil_confirmed] NEUTRAL → ENTAILMENT
  Final Label: ENTAILMENT  |  Confidence: 0.9543

  AUDITING: HDFC Bank (2024)
  Claim (table_normalized):
    No monetary penalties or fines were imposed on the company during the reporting period.
  [Cache HIT]  HDFC Bank||2024  (15 results, skipping Tavily)
  Evidence: 15 raw, 9 passed filter
  [Override applied: rule1_nil_confirmed] NEUTRAL → ENTAILMENT
  Final Label: ENTAILMENT  |  Confidence: 0.9523

  AUDITING: HDFC Bank (2025)
  Claim (no_match):
    INSUFFICIENT_DISCLOSURE
  [Cache HIT]  HDFC Bank||2025  (13 results, skipping Tavily)
  Evidence: 13 raw, 9 passed filter
  Final Label: INSUFFICIENT_DISCLOSURE  |  Confidence: 0.0000

  AUDITING: HUL (2023)
  Claim (no_match):
    INSUFFICIENT_DISCLOSURE


Cannot set gray stroke color because /'P0' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value
Cannot set gray stroke color because /'P2' is an invalid float value


  Claim (table_structured_penalty):
    The company discloses the following monetary penalties imposed during the reporting period: Principle 1 | Reserve Bank o...
  [Cache HIT]  ICICI Bank||2023  (14 results, skipping Tavily)
  Evidence: 14 raw, 6 passed filter
  Final Label: ENTAILMENT  |  Confidence: 0.9626

  AUDITING: ICICI Bank (2024)


...

  Claim (keyword_strict):
    b) The bank failed to comply with the Reserve Bank directions on “Guidelines on management of Intra - group transactions...
  [Cache HIT]  SBI||2024  (15 results, skipping Tavily)
  Evidence: 15 raw, 3 passed filter
  [Override applied: rule2_amount_present] NEUTRAL → CONTRADICTION
  Final Label: CONTRADICTION  |  Confidence: 0.9558

  AUDITING: SBI (2025)


Cannot set gray stroke color because /'P0' is an invalid float value


  Claim (no_match):
    INSUFFICIENT_DISCLOSURE
  [Cache HIT]  SBI||2025  (13 results, skipping Tavily)
  Evidence: 13 raw, 4 passed filter
  Final Label: INSUFFICIENT_DISCLOSURE  |  Confidence: 0.0000

  AUDITING: TCS (2023)


Cannot set gray stroke color because /'P0' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value
Cannot set gray stroke color because /'P2' is an invalid float value
Cannot set gray stroke color because /'P3' is an invalid float value
Cannot set gray stroke color because /'P4' is an invalid float value
Cannot set gray stroke color because /'P5' is an invalid float value
Cannot set gray stroke color because /'P6' is an invalid float value
Cannot set gray stroke color because /'P7' is an invalid float value
Cannot set gray stroke color because /'P8' is an invalid float value
Cannot set gray stroke color because /'P9' is an invalid float value
Cannot set gray stroke color because /'P10' is an invalid float value
Cannot set gray stroke color because /'P11' is an invalid float value
Cannot set gray stroke color because /'P12' is an invalid float value
Cannot set gray stroke color because /'P13' is an invalid float value
Cannot set gray stroke color b

  Claim (keyword_strict):
    If not, provide details of all such non-compliances, in the following format81: Yes, TCS has complied with applicable en...
  [Cache HIT]  TCS||2023  (15 results, skipping Tavily)
  Evidence: 15 raw, 4 passed filter
  [Override applied: rule1_nil_confirmed] NEUTRAL → ENTAILMENT
  Final Label: ENTAILMENT  |  Confidence: 0.7234

  AUDITING: TCS (2024)
  Claim (keyword_strict):
    TCS has complied with applicable environmental law/regulations/guidelines applicable in India. No fine/penalty/action wa...
  [Cache HIT]  TCS||2024  (15 results, skipping Tavily)
  Evidence: 15 raw, 6 passed filter
  [Override applied: rule1_nil_confirmed] NEUTRAL → ENTAILMENT
  Final Label: ENTAILMENT  |  Confidence: 0.9767

  AUDITING: Tata Motors (2023)
  Claim (keyword_strict):
    Details of fines / penalties / punishment / award / compounding fees / settlement amount paid in proceedings (by the ent...
  [Cache HIT]  Tata Motors||2023  (15 results, skipping Tavily)
  Evidence

## Cell 9 — Summary Statistics (Chapter 5 Tables)

In [127]:


df = results_df.copy()


overall = df["Final_Label"].value_counts().reset_index()
overall.columns = ["Classification", "Count"]
overall["Percentage"] = (overall["Count"] / len(df) * 100).round(1)

print("\n===== TABLE 5.1: OVERALL CLASSIFICATION DISTRIBUTION =====")
print(overall.to_string(index=False))


company_pivot = (
    df.groupby(["Company", "Final_Label"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

print("\n===== TABLE 5.2: COMPANY-LEVEL DISTRIBUTION =====")
print(company_pivot.to_string(index=False))


risk_labels  = ["INSUFFICIENT_DISCLOSURE", "CONTRADICTION"]
risk_count   = df["Final_Label"].isin(risk_labels).sum()
risk_pct     = risk_count / len(df) * 100

print(f"\n===== TRANSPARENCY RISK INDICATOR =====")
print(f"  Observations flagged (Insufficient Disclosure + Contradiction): {risk_count} / {len(df)}")
print(f"  Transparency Risk Rate: {risk_pct:.1f}%")


conf_by_label = df.groupby("Final_Label")["Confidence"].mean().round(4)
print("\n===== AVERAGE CONFIDENCE BY LABEL =====")
print(conf_by_label)


===== TABLE 5.1: OVERALL CLASSIFICATION DISTRIBUTION =====
         Classification  Count  Percentage
INSUFFICIENT_DISCLOSURE     21        42.0
             ENTAILMENT     17        34.0
          CONTRADICTION     10        20.0
                NEUTRAL      2         4.0

===== TABLE 5.2: COMPANY-LEVEL DISTRIBUTION =====
      Company  CONTRADICTION  ENTAILMENT  INSUFFICIENT_DISCLOSURE  NEUTRAL
         AXIS              1           0                        0        0
    AXIS Bank              1           0                        1        0
Bharti Airtel              0           0                        3        0
    HDFC Bank              0           2                        1        0
          HUL              0           0                        3        0
   ICICI Bank              0           1                        0        2
          ITC              0           2                        1        0
      Infosys              1           1                        1        0

## Cell 10 — Manual Validation Module

**This cell operationalises Section 5.6 / Chapter 6.6 of the thesis.**

After running the pipeline, manually verify ~8-10 rows by checking the actual PDFs and regulatory sources. Fill in `manual_labels` below, then run this cell to get Accuracy and Precision.

Column guide:
- `company` / `year` — which observation
- `model_label` — what the pipeline predicted (auto-filled from results_df)
- `manual_label` — your verified ground truth after reading the PDF + source
- `notes` — why you assigned that label (for your methodology section)

In [37]:
# ============================================================
# STAGE 6 — MANUAL VALIDATION
# ============================================================
# INSTRUCTIONS:
#   1. Run Cell 9 first to see which observations each label was assigned to.
#   2. Pick a stratified sample: try to include at least 2 CONTRADICTION,
#      2 INSUFFICIENT_DISCLOSURE, 1 ENTAILMENT, 1 NEUTRAL if present.
#   3. For each row below, manually verify by:
#        - Opening the actual BRSR PDF and reading Principle 9
#        - Opening the source URL listed in BRSR_Audit_Detail_Log.csv
#   4. Set manual_label to one of:
#        CONTRADICTION / ENTAILMENT / NEUTRAL / INSUFFICIENT_DISCLOSURE
#   5. Run this cell to get Accuracy and Precision.
# ============================================================

manual_labels = [
    # Fill in your verified labels here. Example format:
    # { "company": "HDFC Bank",    "year": 2025, "manual_label": "CONTRADICTION",        "notes": "RBI penalty notice confirmed on rbi.org.in" },
    # { "company": "Infosys",      "year": 2024, "manual_label": "INSUFFICIENT_DISCLOSURE", "notes": "No penalty sentence found in Principle 9 section" },
    # { "company": "Reliance",     "year": 2023, "manual_label": "CONTRADICTION",        "notes": "CPCB order published June 2023" },
    # { "company": "L&T",          "year": 2023, "manual_label": "INSUFFICIENT_DISCLOSURE", "notes": "Only generic compliance statement found" },
    # { "company": "HUL",          "year": 2024, "manual_label": "NEUTRAL",              "notes": "Evidence found but about a different regulatory matter" },

   

]


if not manual_labels:
    print("No manual labels entered yet. Fill in the manual_labels list above and re-run.")
else:
    val_df = pd.DataFrame(manual_labels)

    
    merged = val_df.merge(
        results_df[["Company", "Year", "Final_Label", "Confidence"]],
        left_on=["company", "year"],
        right_on=["Company", "Year"],
        how="left"
    )

    merged["correct"] = merged["manual_label"] == merged["Final_Label"]

    accuracy = merged["correct"].mean()

    print("\n===== MANUAL VALIDATION RESULTS =====")
    print(merged[["company", "year", "Final_Label", "manual_label", "correct", "Confidence", "notes"]]
          .to_string(index=False))

    print(f"\nOverall Accuracy (pipeline vs. manual): {accuracy:.1%}  ({merged['correct'].sum()}/{len(merged)} correct)")

    
    print("\nPer-class Precision (of pipeline predictions, how many matched manual label):")
    for label in merged["Final_Label"].unique():
        subset = merged[merged["Final_Label"] == label]
        prec   = subset["correct"].mean()
        print(f"  {label:<30}: {prec:.1%}  (n={len(subset)})")

    print("\nNote: Manual validation is a stratified spot-check, not a full ground-truth evaluation.")
    print("Report these figures in Section 5.6 (Methodological Reflection) of your thesis.")

    # Save validation results
    merged.to_csv("BRSR_Manual_Validation.csv", index=False)
    print("Saved to BRSR_Manual_Validation.csv")

No manual labels entered yet. Fill in the manual_labels list above and re-run.


## Cell 11 — Methodological Notes for Thesis Chapter 5 / 6

### What changed from the original notebook, and why

| Issue | Original | This version |
|---|---|---|
| Claim extraction | Keyword-only; returned hardcoded fallback string on miss → inflated Contradiction rate | Keyword + Semantic similarity; `INSUFFICIENT_DISCLOSURE` only when both fail |
| Evidence quality | All 5 Tavily snippets passed to NLI unfiltered | Filter: snippet must contain regulatory keyword AND FY year string |
| NLI decision | Raw argmax (lowest confidence wins) | Calibrated thresholds (0.75) — labels only assigned if confidence sufficient |
| Output structure | Scattered across cells; no single DataFrame | Structured: Company / Year / Claim / Label / Confidence / Source per row |
| Validation | None | Manual validation module with Accuracy + Precision computation |

### Interpreting results in your thesis

- **INSUFFICIENT_DISCLOSURE** → No machine-detectable penalty statement in the BRSR (greenhushing-consistent)
- **CONTRADICTION** ≥ 0.75 confidence → Semantic tension with verified external source (greenwashing-consistent)
- **ENTAILMENT** ≥ 0.75 confidence → Disclosure aligns with external evidence (honest disclosure)
- **NEUTRAL** → Evidence found but insufficient confidence for either contradiction or entailment
- **NO_EVIDENCE_FOUND** → Evidence retrieved but none passed the quality filter for the specified FY

### Limitations to acknowledge
1. Calibrated thresholds (0.75) are heuristic, not derived from a labelled BRSR dataset.
2. Semantic extraction (Phase B) uses a general embedding model not fine-tuned on regulatory text.
3. Year-filter may exclude valid evidence if news articles use inconsistent FY notation.
4. Manual validation spot-check (Cell 10) does not replace full ground-truth evaluation.